# Saheli — GGUF Conversion (Kaggle)

Pulls the LoRA adapter from HuggingFace, merges it into base Gemma 4 E4B via Unsloth,
and exports Q4_K_M GGUF for llama.cpp offline inference.

**Runtime:** Accelerator → GPU T4 (Settings panel on the right).  
**Disk needed:** ~5 GB in `/kaggle/working` (GGUF output only — base model caches elsewhere).  
**HF_TOKEN secret:** Add it under Add-ons → Secrets → `HF_TOKEN` with read+write.

## Step 1 — Configure

In [ ]:
import os

HF_USER      = 'sriramarivazhagan'
ADAPTER_REPO = f'{HF_USER}/saheli-gemma4-e4b'
QUANT        = 'q4_k_m'

ADAPTER_DIR  = '/kaggle/working/adapter'
GGUF_OUT_DIR = '/kaggle/working/gguf'

PUSH_GGUF_TO_HF = True   # push finished GGUF back to same HF repo

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

print('Adapter repo :', ADAPTER_REPO)
print('GGUF output  :', GGUF_OUT_DIR)

## Step 2 — Install Unsloth

⚠️ After this cell finishes, **restart the kernel** (Run → Restart & Clear Output),
then continue from Step 3.

In [ ]:
!pip install -q unsloth
!pip install -q --upgrade --no-deps unsloth
!pip install -q huggingface_hub hf_transfer sentencepiece

import torch, unsloth
print('torch   :', torch.__version__)
print('unsloth :', unsloth.__version__)
print('CUDA    :', torch.cuda.is_available())
print()
print('Now restart the kernel, then run Steps 3 onward.')

## Step 3 — Authenticate with HuggingFace

Add your token under **Add-ons → Secrets → HF_TOKEN** (read+write scope).

In [ ]:
import os
from huggingface_hub import login

# Kaggle secrets are exposed as environment variables
HF_TOKEN = os.environ.get('HF_TOKEN', '')
assert HF_TOKEN, (
    'HF_TOKEN not found.\n'
    'Add it under Add-ons → Secrets → HF_TOKEN, '
    'then enable it for this notebook.'
)
login(token=HF_TOKEN)
print('Logged in to HuggingFace.')

## Step 4 — Download LoRA adapter from HuggingFace

In [ ]:
import os
from huggingface_hub import snapshot_download

# Re-declare constants (needed after kernel restart)
HF_USER      = 'sriramarivazhagan'
ADAPTER_REPO = f'{HF_USER}/saheli-gemma4-e4b'
ADAPTER_DIR  = '/kaggle/working/adapter'
GGUF_OUT_DIR = '/kaggle/working/gguf'
QUANT        = 'q4_k_m'
PUSH_GGUF_TO_HF = True

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

# Only download the adapter (~150 MB). Unsloth resolves the base from adapter_config.json.
snapshot_download(ADAPTER_REPO, local_dir=ADAPTER_DIR)
print('Adapter dir :', ADAPTER_DIR)
!ls -lh {ADAPTER_DIR}
!df -h /kaggle/working

## Step 5 — Load model + adapter and export GGUF Q4_K_M

Unsloth reads `adapter_config.json`, downloads the base model to its own cache
(outside `/kaggle/working`), loads in 4-bit, merges the adapter, and writes the
quantised GGUF directly — no fp16 intermediate needed in working storage.

In [ ]:
import os, torch
from unsloth import FastModel

os.makedirs(GGUF_OUT_DIR, exist_ok=True)

print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

print('\nLoading base + LoRA adapter via Unsloth (4-bit)...')
model, tokenizer = FastModel.from_pretrained(
    model_name     = ADAPTER_DIR,   # local adapter dir; Unsloth auto-fetches base
    max_seq_length = 512,           # keep small — we only need export, not inference
    load_in_4bit   = True,
)

print('\nExporting GGUF Q4_K_M (takes ~15 min)...')
model.save_pretrained_gguf(
    GGUF_OUT_DIR,
    tokenizer,
    quantization_method = QUANT,
)

print('\nOutput files:')
for f in os.listdir(GGUF_OUT_DIR):
    size = os.path.getsize(f'{GGUF_OUT_DIR}/{f}') / 1e9
    print(f'  {f}  ({size:.2f} GB)')

!df -h /kaggle/working

## Step 6 — Locate the GGUF file

In [ ]:
import os, glob

matches = glob.glob(f'{GGUF_OUT_DIR}/*.gguf')
assert matches, f'No GGUF found in {GGUF_OUT_DIR} — Step 5 may have failed'
Q_GGUF = matches[0]
print('GGUF file:', Q_GGUF)
print(f'Size     : {os.path.getsize(Q_GGUF)/1e9:.2f} GB')

## Step 7 — Quick sanity check

Loads the GGUF via llama-cpp-python and runs 5 triage cases.

In [ ]:
!pip install -q llama-cpp-python

import re, glob, os
from llama_cpp import Llama

# Find the GGUF — Unsloth appends _gguf to the output dir name
matches = glob.glob(f'{GGUF_OUT_DIR}_gguf/*Q4*.gguf') or glob.glob(f'{GGUF_OUT_DIR}/*Q4*.gguf')
assert matches, f'No Q4 GGUF found under {GGUF_OUT_DIR} or {GGUF_OUT_DIR}_gguf'
Q_GGUF = matches[0]
print('Testing:', Q_GGUF, f'({os.path.getsize(Q_GGUF)/1e9:.2f} GB)')

llm = Llama(
    model_path   = Q_GGUF,
    chat_format  = 'gemma',
    n_ctx        = 2048,
    n_gpu_layers = -1,   # offload all layers to GPU
    verbose      = False,
)

# Must match the training system prompt exactly
SYSTEM_PROMPT = (
    'You are Saheli, a maternal health assistant for ASHA workers. '
    'Given the patient\'s symptoms and vitals, output a JSON tool call to '
    'assess_danger_signs followed by a short RED/YELLOW/GREEN recommendation '
    'in plain language. Use WHO Antenatal Care guidelines.'
)

CASES = [
    ('Patient is 32 weeks pregnant and has severe headache and blurred vision. BP 150/100.', 'RED'),
    ('28-week patient, baby has not moved all day.',                                         'RED'),
    ('36-week patient, mild ankle swelling. BP 130/85.',                                     'YELLOW'),
    ('ASHA visit: 24w, feeling very weak, haemoglobin 6.5.',                                 'YELLOW'),
    ('16-week patient, some nausea in the mornings.',                                        'GREEN'),
]

correct = 0
for user_msg, expected in CASES:
    resp = llm.create_chat_completion(
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': user_msg},
        ],
        max_tokens  = 200,
        temperature = 0.1,
    )
    out  = resp['choices'][0]['message']['content']
    m    = re.search(r'\b(RED|YELLOW|GREEN)\b', out.upper())
    pred = m.group(1) if m else '?'
    ok   = pred == expected
    correct += int(ok)
    print(f"[{'PASS' if ok else 'FAIL'}] expected {expected:<6} got {pred:<6} | {user_msg[:60]}")
    print(f'  {out[:160]}')

print(f'\nSanity: {correct}/{len(CASES)} correct')
if correct == 0:
    print('\nWARNING: 0/5 correct suggests the adapter was NOT merged.')
    print('Check Step 5 output for: "Unsloth: Model is not a PEFT model"')

## Step 8 — Push GGUF to HuggingFace

Uploads to the same repo as the adapter so adapter + GGUF live together.

In [ ]:
if PUSH_GGUF_TO_HF:
    from huggingface_hub import upload_file
    upload_file(
        path_or_fileobj = Q_GGUF,
        path_in_repo    = os.path.basename(Q_GGUF),
        repo_id         = ADAPTER_REPO,
        repo_type       = 'model',
    )
    print(f'Pushed → https://huggingface.co/{ADAPTER_REPO}/blob/main/{os.path.basename(Q_GGUF)}')
else:
    print('PUSH_GGUF_TO_HF=False — skipped.')

## Step 9 — Download GGUF to your computer

On Kaggle the output files are available in the **Output** tab on the right panel
after the session ends. You can also download directly with the cell below.

In [ ]:
print('GGUF is saved at:', Q_GGUF)
print()
print('To download: open the Output tab (right panel) → find the file → Download.')
print('Or push to HF (Step 8) and pull it from there.')
print()
print('Next steps on your local machine:')
print('  1. Move the .gguf to saheli/models/gemma-4-e4b-q4_k_m.gguf')
print('  2. Update config/settings.py  MODEL_PATH to point at it')
print('  3. python finetune/evaluate_model.py   (three-row benchmark)')